In [1]:
import torch, ultralytics
import numpy as np
import cv2
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
print("ultralytics", ultralytics.__version__)
assert torch.cuda.is_available(), "no GPU - Runtime > Change runtime type > GPU"

torch 2.7.1+cu118 | cuda True
ultralytics 8.4.139


In [2]:
# --- data -------------------------------------------------------------------
# Leave DRIVE_VIDEO_DIR empty to search your Drive for the videos by name -- a
# share link gives a folder id, not a path, so the path cannot be derived from it.
# Set it explicitly only if the search picks the wrong copy.
DRIVE_VIDEO_DIR = ""
VIDEO_GLOB = "29-gameplay-*.mp4"
USE_DRIVE = True          # False -> browser upload widget instead

# --- training ---------------------------------------------------------------
N_TRAIN, N_VAL = 4000, 600   # synthetic scenes
EPOCHS = 60
BATCH = 32                   # drop to 16 if the T4 runs out of memory
IMGSZ = 512                  # must stay equal to CANON_SIZE

# --- reconstruction ---------------------------------------------------------
SAMPLE_FPS = 4.0          # how often to look at the video
CONF = 0.30               # detector threshold; voting tolerates a permissive value

# --- rules ------------------------------------------------------------------
# Who led the very first trick. The video cannot tell you this; in 29 it is the
# bid winner. Every later leader is derived (winner of a trick leads the next).
FIRST_LEADER = 1
DEFAULT_TRUMP_CODE = "S"  # S/H/D/C. Trump is not read off the table yet.
BID_TEAM, BID = "A", 16   # team A = players 1 & 3, team B = 2 & 4

In [3]:
VIDEO1_PATH="Z:\\coding\\PR project\\29-card-game\\29-gameplay-1 - Converted.mp4"
VIDEO2_PATH="Z:\\coding\\PR project\\29-card-game\\29-gameplay-2.mp4"
TRIMMED_PATH="Z:\\coding\\PR project\\29-card-game\\trimmed.mp4"

In [9]:
from pathlib import Path
# just sorts the video according to the path
VIDEOS = sorted(str(path) for path in Path("29-card-game").glob("*.mp4"))
TRAIN_DIR="Z:\\coding\\PR project\\dataset\\train"
VAL_DIR="Z:\\coding\\PR project\\dataset\\val"
OUTPUT_DIR="Z:\\coding\\PR project\\dataset"

### Training attempt - 01

* Trying to convert the video into images
* taking a sample of 4 fps and saving all it in \train folder
* then annotation needs to be done
* Working on only the first video

In [12]:
sample_fps = 4.0
saved = { "train": 0, "val": 0}

cap = cv2.VideoCapture(VIDEO1_PATH)

video_fps = cap.get(cv2.CAP_PROP_FPS)
video_fps

frame_step = max(1, round(video_fps / sample_fps))
print(frame_step)

frame_index = 0
saved_from_video = 0
split = "train" # putting every frame into the training set now before labelling

while True:
        ok, frame = cap.read()
        if not ok:
            print("Video reading has ended, no more frames can be read")
            break

        if frame_index % frame_step == 0: # stops the look when there are no more frames
            filename = (
                f"{Path(VIDEO2_PATH).stem}"
                f"_frame_{frame_index:08d}.jpg"
            ) # naming should be something like '29-gameplay-1_frame_00000120.jpg'
            output_path =  OUTPUT_DIR + "/" + filename

            cv2.imwrite(str(output_path), frame)
            saved[split] += 1
            saved_from_video += 1

        frame_index += 1

cap.release()
print(f"{Path(VIDEO1_PATH).name}: {saved_from_video} frames -> {split}")

print(f"Training images:   {saved['train']}")
print(f"Validation images: {saved['val']}")

8
Video reading has ended, no more frames can be read
29-gameplay-1 - Converted.mp4: 1546 frames -> train
Training images:   1546
Validation images: 0


In [ ]:
# CANON_SIZE = 512
# CANON_RADIUS = 230.0
# CANON_CENTER = (CANON_SIZE / 2.0, CANON_SIZE / 2.0)

# def _table_disc_mask():
#     """The whole table top, not just the play area.

#     Emptiness is scored over the entire table on purpose. Any real face-up card
#     left in a background is an *unlabelled* card in a training image, which teaches
#     the detector to ignore exactly what it is meant to find -- and that applies
#     wherever on the table it sits, not only in the middle. The face-down hand piles
#     are always present and so contribute a near-constant offset that ranking
#     absorbs harmlessly.
#     """
#     yy, xx = np.mgrid[0:CANON_SIZE, 0:CANON_SIZE]
#     return np.hypot(xx - CANON_CENTER[0], yy - CANON_CENTER[1]) < 0.98 * CANON_RADIUS

# def harvest_backgrounds(video_paths, tracker_cls, max_per_video=60, sample_every=2.0):
#     """Collect canonical table crops whose play area is as empty as possible.

#     Emptiness is *ranked*, not thresholded. The cane weave's own highlights read as
#     bright and desaturated, so even a bare table scores ~8% "card-like" pixels in
#     the table -- any absolute cutoff either takes everything or nothing. Taking
#     the lowest-scoring frames per video needs no tuned constant and adapts to
#     whatever the lighting happens to be.

#     Two passes, so only the selected frames are ever held in memory: score first,
#     then re-read the winners using the table fit recorded alongside each score.
#     """
#     play = _table_disc_mask()
#     backgrounds = []

#     for path in video_paths:
#         cap = cv2.VideoCapture(path)
#         print(cap)
#         fps = max(cap.get(cv2.CAP_PROP_FPS), 1.0)
#         step = max(1, int(fps * sample_every))

#         tracker = tracker_cls()
#         scored = []
#         i = -1
#         while True:  # sequential decode; seeking per sample is far slower
#             if not cap.grab():
#                 break
#             i += 1
#             if i % step:
#                 continue
#             ok, frame = cap.retrieve()
#             if not ok:
#                 break
#             fit = tracker.update(frame)
#             if fit is None:
#                 continue
#             hsv = cv2.cvtColor(fit.warp(frame), cv2.COLOR_BGR2HSV)
#             print(hsv)
#             cardish = (hsv[:, :, 2] > 150) & (hsv[:, :, 1] < 60)
#             scored.append((float((cardish & play).sum() / play.sum()), i, fit))

#         # Only the winners are re-read, so peak memory stays at max_per_video frames.
#         scored.sort(key=lambda t: t[0])
#         for _, i, fit in scored[:max_per_video]:
#             cap.set(cv2.CAP_PROP_POS_FRAMES, i)
#             ok, frame = cap.read()
#             if ok:
#                 backgrounds.append(fit.warp(frame))
#         cap.release()

#     return backgrounds

In [ ]:
# from card_faces import load_gallery
# from synthetic_canon import make_scene
# from p29.vision.registration import TableTracker

# # GALLERY = load_gallery("data/reference")  # real scans win when present
# BACKGROUNDS = harvest_backgrounds(VIDEOS, TableTracker, max_per_video=60)

# # print(BACKGROUNDS)
# # print("empty-table backgrounds harvested:", len(BACKGROUNDS))
# # assert BACKGROUNDS, "no clean backgrounds found - check registration above"

< cv2.VideoCapture 000001BB5F4E3D70>
[[[  0   0   0]
  [  0   0   0]
  [  0   0   0]
  ...
  [126  80  16]
  [126  85  15]
  [126  91  14]]

 [[  0   0   0]
  [  0   0   0]
  [  0   0   0]
  ...
  [126  80  16]
  [126  85  15]
  [126  85  15]]

 [[  0   0   0]
  [  0   0   0]
  [  0   0   0]
  ...
  [126  80  16]
  [126  85  15]
  [126  85  15]]

 ...

 [[  0   0   0]
  [  0   0   0]
  [  0   0   0]
  ...
  [156 120  34]
  [156 120  34]
  [156 120  34]]

 [[  0   0   0]
  [  0   0   0]
  [  0   0   0]
  ...
  [156 120  34]
  [156 124  33]
  [156 124  33]]

 [[  0   0   0]
  [  0   0   0]
  [  0   0   0]
  ...
  [156 120  34]
  [156 124  33]
  [156 124  33]]]
[[[  0   0   0]
  [  0   0   0]
  [  0   0   0]
  ...
  [124 136  15]
  [124 136  15]
  [128 136  15]]

 [[  0   0   0]
  [  0   0   0]
  [  0   0   0]
  ...
  [120 136  15]
  [120 136  15]
  [120 136  15]]

 [[  0   0   0]
  [  0   0   0]
  [  0   0   0]
  ...
  [120 136  15]
  [120 136  15]
  [120 136  15]]

 ...

 [[  0   0   0]